# Macro indicator extraction for GAN conditioning

- load the Bloomberg export
- select a **small macro set**
- resample to **weekly**
- apply simple transformations
- export a clean macro feature matrix for the GAN

**NOT** using description-based search, Random Forest, or RFE.
We are selecting the macro indicators that are the best fit from the available Bloomberg series and that have usable history.


## Chosen indicators

These are the indicators selected for the first pass because they are broad, interpretable, and useful as external conditioning variables for generating synthetic data for the 30 assets.

### Labor / growth
- **INJCJC Index** — Initial Jobless Claims

### Policy / money market
- **FDFD Index** — Effective Fed Funds
- **US0003M Index** — USD 3M LIBOR

### Rates curve
- **USSWAP2 CMPN Curncy** — USD 2Y Swap
- **USSWAP10 CMPN Curncy** — USD 10Y Swap

### Broad commodity pressure
- **CRY Index** — CRB Commodity Index

### Credit / risk regime
- **IBOXUMAE CBIN Curncy** — Markit CDX IG
- **IBOXHYAE CBIN Curncy** — Markit CDX HY

In [45]:
import pandas as pd
import numpy as np

In [46]:
# File names
DATA_FILE = "../data/macro/bloomberg_data.csv"
DESC_FILE = "../data/macro/bloomberg_desc.csv" 

In [47]:
# ---------------------------------------------------
# 1. Load Bloomberg data
# ---------------------------------------------------
preamble = pd.read_csv(DATA_FILE, header=None, nrows=6, low_memory=False)

tickers = preamble.iloc[3, 1:].ffill().tolist()
fields = preamble.iloc[5, 1:].tolist()

cols = pd.MultiIndex.from_arrays([tickers, fields], names=["ticker", "field"])

raw = pd.read_csv(
    DATA_FILE,
    skiprows=6,
    header=None,
    index_col=0,
    parse_dates=True,
    low_memory=False
)

raw.index.name = "Date"
raw.columns = cols
raw = raw.sort_index()

# Keep only PX_LAST
px = raw.xs("PX_LAST", axis=1, level="field").copy()
px.columns.name = "ticker"

# Remove duplicate ticker columns
px = px.loc[:, ~px.columns.duplicated()]

print("PX_LAST shape:", px.shape)
print("Date range:", px.index.min(), "to", px.index.max())

PX_LAST shape: (9420, 292)
Date range: 1990-01-01 00:00:00 to 2026-02-06 00:00:00


In [48]:
# ---------------------------------------------------
# 2. Load descriptions
# ---------------------------------------------------
desc = pd.read_csv(DESC_FILE, header=None)
desc.columns = ["ticker", "name", "currency", "note"]
desc["ticker"] = desc["ticker"].astype(str).str.strip()

desc_map = desc.set_index("ticker")["name"].to_dict()

In [49]:

# ---------------------------------------------------
# 3. Hand-picked macro indicators
# ---------------------------------------------------
selected_tickers = [
    "INJCJC Index",            # Initial Jobless Claims
    "FDFD Index",              # Effective Fed Funds
    "US0003M Index",           # USD 3M LIBOR
    "USSWAP2 CMPN Curncy",     # USD 2Y Swap
    "USSWAP10 CMPN Curncy",    # USD 10Y Swap
    "CRY Index",               # CRB Commodity Index
    "IBOXUMAE CBIN Curncy",    # Markit CDX IG
    "IBOXHYAE CBIN Curncy",    # Markit CDX HY
]


available_tickers = [t for t in selected_tickers if t in px.columns]
missing_tickers = [t for t in selected_tickers if t not in px.columns]

print("Available selected tickers:", len(available_tickers))
print(available_tickers)

if missing_tickers:
    print("Missing selected tickers:")
    print(missing_tickers)


Available selected tickers: 8
['INJCJC Index', 'FDFD Index', 'US0003M Index', 'USSWAP2 CMPN Curncy', 'USSWAP10 CMPN Curncy', 'CRY Index', 'IBOXUMAE CBIN Curncy', 'IBOXHYAE CBIN Curncy']


In [50]:
macro_raw = px[available_tickers].copy()

macro_info = pd.DataFrame({
    "ticker": available_tickers,
    "description": [desc_map.get(t, "") for t in available_tickers],
    "missing_pct_daily": [macro_raw[t].isna().mean() for t in available_tickers]
}).sort_values("missing_pct_daily")

macro_info

,ticker,description,missing_pct_daily
2,US0003M Index,ICE LIBOR USD 3M,0.068153
3,USSWAP2 CMPN Curncy,US DOLLAR SWAP 2 YR,0.074098
4,USSWAP10 CMPN Curncy,US DOLLAR SWAP 10 YR,0.075053
5,CRY Index,FTSE/CoreCommodity CRB Excess,0.141189
1,FDFD Index,Fed Funds,0.175584
6,IBOXUMAE CBIN Curncy,MARKIT CDX IG,0.418577
7,IBOXHYAE CBIN Curncy,MARKIT CDX HY,0.529299
0,INJCJC Index,Initial Jobless Claims SA,0.800106


## Rename columns to simpler names

This makes the conditioning matrix easier to use later in the GAN pipeline.


In [51]:

rename_map = {
    "INJCJC Index": "initial_jobless_claims",
    "FDFD Index": "fed_funds",
    "US0003M Index": "usd_3m_libor",
    "USSWAP2 CMPN Curncy": "usd_2y_swap",
    "USSWAP5 CMPN Curncy": "usd_5y_swap",
    "CRY Index": "crb_commodity_index",
    "IBOXUMAE CBIN Curncy": "cdx_ig",
    "IBOXHYAE CBIN Curncy": "cdx_hy",
}

macro_raw = macro_raw.rename(columns=rename_map)
macro_raw.head()


ticker,initial_jobless_claims,fed_funds,usd_3m_libor,usd_2y_swap,USSWAP10 CMPN Curncy,crb_commodity_index,cdx_ig,cdx_hy
Date,,,,,,,,
1990-01-01,NaN,NaN,8.3750,NaN,NaN,NaN,NaN,NaN
1990-01-02,NaN,NaN,8.3750,NaN,NaN,NaN,NaN,NaN
1990-01-03,NaN,NaN,8.3750,NaN,NaN,NaN,NaN,NaN
1990-01-04,NaN,NaN,8.3125,8.44,8.89,NaN,NaN,NaN
1990-01-05,355.0,NaN,8.3750,8.43,8.92,NaN,NaN,NaN


## Weekly alignment

Macro indicators are easier to use as conditioning variables if everything is on one frequency.

In [52]:
macro_monthly = macro_raw.resample("W-FRI").last().ffill()

summary_monthly = pd.DataFrame({
    "feature": macro_monthly.columns,
    "missing_pct_monthly": macro_monthly.isna().mean().values,
    "first_valid": [macro_monthly[c].first_valid_index() for c in macro_monthly.columns],
    "last_valid": [macro_monthly[c].last_valid_index() for c in macro_monthly.columns]
}).sort_values("missing_pct_monthly")

summary_monthly

,feature,missing_pct_monthly,first_valid,last_valid
0,initial_jobless_claims,0.000000,1990-01-05,2026-02-06
2,usd_3m_libor,0.000000,1990-01-05,2026-02-06
3,usd_2y_swap,0.000000,1990-01-05,2026-02-06
4,USSWAP10 CMPN Curncy,0.000000,1990-01-05,2026-02-06
5,crb_commodity_index,0.110934,1994-01-07,2026-02-06
1,fed_funds,0.141720,1995-02-17,2026-02-06
6,cdx_ig,0.408174,2004-10-01,2026-02-06
7,cdx_hy,0.493631,2007-11-02,2026-02-06



## Simple transformations

- Rates and credit spreads -> first difference
- Claims and commodity index -> percent change


In [53]:

def transform_feature(s):
    name = s.name

    diff_series = [
        "FDFD Index",
        "US0003M Index",
        "USSWAP2 CMPN Curncy",
        "USSWAP10 CMPN Curncy",
        "IBOXUMAE CBIN Curncy",
        "IBOXHYAE CBIN Curncy",
    ]

    pct_series = [
        "INJCJC Index",
        "CRY Index",
    ]

    if name in diff_series:
        return s.diff()

    if name in pct_series:
        return s.pct_change()

    return s.copy()


In [54]:
macro_features = macro_monthly.apply(transform_feature)
macro_features = macro_features.replace([np.inf, -np.inf], np.nan)

# Keep a recent usable sample window
macro_features = macro_features.loc["2005-01-31":].copy()

# Drop rows that are fully missing
macro_features = macro_features.dropna(how="all")

macro_features.head()

ticker,initial_jobless_claims,fed_funds,usd_3m_libor,usd_2y_swap,USSWAP10 CMPN Curncy,crb_commodity_index,cdx_ig,cdx_hy
Date,,,,,,,,
2005-02-04,307.0,2.5625,2.77000,3.645,-0.059,279.69,44.500,NaN
2005-02-11,308.0,2.4375,2.79438,3.679,0.001,284.84,45.042,NaN
2005-02-18,318.0,2.5000,2.85000,3.813,0.183,290.46,44.104,NaN
2005-02-25,314.0,2.5000,2.91000,3.880,0.007,301.29,44.146,NaN
2005-03-04,333.0,2.5000,2.95875,3.923,0.064,310.08,43.875,NaN


In [55]:
feature_quality = pd.DataFrame({
    "feature": macro_features.columns,
    "missing_pct": macro_features.isna().mean().values,
    "std": macro_features.std().values
}).sort_values(["missing_pct", "std"], ascending=[True, False])

feature_quality

,feature,missing_pct,std
0,initial_jobless_claims,0.000000,384.498591
5,crb_commodity_index,0.000000,63.613740
6,cdx_ig,0.000000,34.119193
2,usd_3m_libor,0.000000,2.024800
1,fed_funds,0.000000,1.962661
3,usd_2y_swap,0.000000,1.854187
4,USSWAP10 CMPN Curncy,0.000000,0.111774
7,cdx_hy,0.130356,6.775255


## Keep the final feature set

Remove columns that are too incomplete.


In [56]:
MAX_MISSING = 0.20

final_features = feature_quality.loc[
    feature_quality["missing_pct"] <= MAX_MISSING, "feature"
].tolist()

macro_final = macro_features[final_features].dropna()

print("Final shape:", macro_final.shape)
print("Final features:")
print(macro_final.columns.tolist())

macro_final.head()

Final shape: (954, 8)
Final features:
['initial_jobless_claims', 'crb_commodity_index', 'cdx_ig', 'usd_3m_libor', 'fed_funds', 'usd_2y_swap', 'USSWAP10 CMPN Curncy', 'cdx_hy']


ticker,initial_jobless_claims,crb_commodity_index,cdx_ig,usd_3m_libor,fed_funds,usd_2y_swap,USSWAP10 CMPN Curncy,cdx_hy
Date,,,,,,,,
2007-11-02,327.0,353.57,69.722,4.86500,4.000,4.3995,-0.0400,96.768
2007-11-09,333.0,354.54,78.156,4.87938,4.500,4.2290,-0.0430,95.408
2007-11-16,332.0,349.43,76.999,4.94875,4.500,4.2005,-0.0575,95.163
2007-11-23,352.0,354.29,86.880,5.04000,4.625,4.0210,-0.1395,93.574
2007-11-30,344.0,339.84,75.973,5.13125,4.500,3.8950,-0.1740,95.609


In [ ]:
# Correlation check
corr = macro_final.corr().round(2)
corr

ticker,initial_jobless_claims,crb_commodity_index,cdx_ig,usd_3m_libor,fed_funds,usd_2y_swap,USSWAP10 CMPN Curncy,cdx_hy
ticker,,,,,,,,
initial_jobless_claims,1.00,-0.22,0.20,-0.21,-0.25,-0.27,-0.02,-0.26
crb_commodity_index,-0.22,1.00,0.25,0.24,0.20,0.29,0.02,-0.30
cdx_ig,0.20,0.25,1.00,-0.17,-0.30,-0.17,-0.12,-0.88
usd_3m_libor,-0.21,0.24,-0.17,1.00,0.97,0.97,-0.02,0.07
fed_funds,-0.25,0.20,-0.30,0.97,1.00,0.94,-0.01,0.18
usd_2y_swap,-0.27,0.29,-0.17,0.97,0.94,1.00,0.02,0.06
USSWAP10 CMPN Curncy,-0.02,0.02,-0.12,-0.02,-0.01,0.02,1.00,0.06
cdx_hy,-0.26,-0.30,-0.88,0.07,0.18,0.06,0.06,1.00


In [ ]:
# Export final features
macro_final.to_csv("../data/macro/macro_conditioning_features.csv")

print("Saved: macro_conditioning_features.csv")

Saved: macro_conditioning_features.csv
